# Serve Smart: Multi-Class Military Object Detection System

This notebook builds an end‑to‑end **YOLO-based object detection system** for a 12‑class military object dataset: training, validation, inference, and packaging YOLO‑format predictions for submission.

**What you'll get**
- Trained YOLO model (Ultralytics)
- Validation metrics (mAP@50, precision/recall, confusion matrix)
- YOLO‑TXT predictions for each test image (with confidence)
- Zipped results + lightweight report template

> **Note:** If you're on **Google Colab**, choose `Runtime → Change runtime type → T4 GPU` for best results.


## 1) Setup

In [1]:

# If running locally, uncomment and run the next line to install requirements
!pip install ultralytics opencv-python matplotlib roboflow -q

import os, sys, zipfile, glob, shutil, json, time, random
from pathlib import Path

import matplotlib.pyplot as plt

try:
    import ultralytics
    from ultralytics import YOLO
except Exception as e:
    print("If Ultralytics is not installed, run: pip install ultralytics")
    raise

print("Ultralytics version:", ultralytics.__version__)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


WARNING Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\adars\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics version: 8.3.235


## 2) Dataset

The PDF specifies a Google Drive dataset with folders:

```
military_object_dataset/
├── train/images/
├── train/labels/
├── val/images/
├── val/labels/
├── test/images/
└── military_dataset.yaml
```

Paste the **public** Drive URL (or mount Drive and point to the folder). If you already have the data locally, set `DATASET_DIR` directly.


In [2]:

# === Configure dataset location ===
# Option A: Provide a public Google Drive download link (file or zipped folder).
DATASET_DRIVE_URL =""  # e.g., "https://drive.google.com/uc?id=<FILE_ID>" or "https://drive.google.com/file/d/<FILE_ID>/view"

# Option B: If dataset is already present, set the directory path directly.
DATASET_DIR = "C:/Users/adars/OneDrive/Desktop/military_object_dataset"  # e.g., "/content/military_object_dataset"

# If your dataset is a zip file (downloaded via URL or uploaded), set ZIP path and we'll unzip it.
DATASET_ZIP = ""  # e.g., "/content/military_object_dataset.zip"

# Helper: Attempt to fetch via gdown if a Drive URL is provided (Colab/Kaggle often have gdown)
def maybe_download_with_gdown(url, out_path):
    try:
        import gdown
    except:
        os.system("pip install gdown -q")
        import gdown
    print("Downloading with gdown...")
    gdown.download(url, out_path, fuzzy=True)

# Resolve dataset directory
root = Path.cwd()
resolved_dataset_dir = None

if DATASET_DIR and Path(DATASET_DIR).exists():
    resolved_dataset_dir = Path(DATASET_DIR)
    print("Using local dataset:", resolved_dataset_dir)
elif DATASET_ZIP and Path(DATASET_ZIP).exists():
    print("Unzipping dataset:", DATASET_ZIP)
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
        zf.extractall(root)
    # Heuristic: find top-level extracted folder containing 'train/images'
    candidates = list(root.glob("**/train/images"))
    if candidates:
        resolved_dataset_dir = candidates[0].parents[1]
        print("Detected dataset directory:", resolved_dataset_dir)
elif DATASET_DRIVE_URL:
    out_zip = root / "dataset_download.zip"
    maybe_download_with_gdown(DATASET_DRIVE_URL, str(out_zip))
    if out_zip.exists():
        print("Unzipping downloaded archive...")
        with zipfile.ZipFile(out_zip, 'r') as zf:
            zf.extractall(root)
        candidates = list(root.glob("**/train/images"))
        if candidates:
            resolved_dataset_dir = candidates[0].parents[1]
            print("Detected dataset directory:", resolved_dataset_dir)
        else:
            print("Could not auto-detect dataset after unzip. Please set DATASET_DIR manually.")
    else:
        print("Download failed. Please verify the Drive URL or set DATASET_DIR.")
else:
    print("Please set DATASET_DIR or provide DATASET_DRIVE_URL / DATASET_ZIP.")

if resolved_dataset_dir:
    # Sanity checks
    expected = ["train/images", "train/labels", "val/images", "val/labels", "test/images"]
    for sub in expected:
        print(sub, "OK" if (resolved_dataset_dir / sub).exists() else "MISSING")


Using local dataset: C:\Users\adars\OneDrive\Desktop\military_object_dataset
train/images OK
train/labels OK
val/images OK
val/labels OK
test/images OK


## 3) Configure YOLO YAML

In [3]:

# Try to locate the dataset YAML automatically; otherwise specify below.
yaml_path = None
candidates = list((resolved_dataset_dir or Path.cwd()).glob("**/*.yaml"))
for c in candidates:
    if "military" in c.name.lower():
        yaml_path = c
        break

if yaml_path is None:
    # Fallback: set manually
    yaml_path = "/content/military_object_dataset/military_dataset.yaml"  # e.g., f"{resolved_dataset_dir}/military_dataset.yaml"

print("YAML path:", yaml_path)
if not yaml_path or (isinstance(yaml_path, str) and not yaml_path):
    print("⚠️  If YAML path is empty, set it manually above.")


YAML path: C:\Users\adars\OneDrive\Desktop\military_object_dataset\military_dataset.yaml


In [4]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu13
import torch
# ... (run imports again)

print(f"CUDA is available: {torch.cuda.is_available()}")

Looking in indexes: https://download.pytorch.org/whl/cu13CUDA is available: True




[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install --upgrade ultralytics


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 4) Train YOLO

- Start with **YOLOv8m** for accuracy; optionally try **YOLOv8s** for speed.

- Use strong augmentations for robustness (mosaic, HSV, rotations).

- Early stopping helps prevent overfitting.


In [6]:
# Choose a base model: 'yolov8m.pt'
BASE_MODEL = "yolov8m.pt"
PROJECT_NAME = "serve_smart_yolov8"
EPOCHS = 100
IMG_SIZE = 640
BATCH = 16 

# Assuming 'yaml_path' is defined and correct
assert yaml_path, "Please set 'yaml_path' to your dataset YAML file."
print("Training with:", BASE_MODEL)

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    name=PROJECT_NAME,
    project="runs/detect",
    optimizer="auto",
    patience=25,
    cos_lr=True,
    augment=True,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    
    # CRITICAL FIXES
    device=0,      # Ensure GPU is used
    workers=1      # Avoids Windows multiprocessing freeze
)
print("Training complete.")

Training with: yolov8m.pt
Ultralytics 8.3.235  Python-3.13.3 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\adars\OneDrive\Desktop\military_object_dataset\military_dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=serve_smart_yolov85, nbs=64, nms=False,

## 5) Validate

In [3]:
import os, glob, torch, cv2
from ultralytics import YOLO
from pathlib import Path

# paths
best_weights = Path(r"C:\Users\adars\Downloads\runs\detect\serve_smart_yolov85\weights\best.pt")
data_yaml    = Path(r"C:\Users\adars\OneDrive\Desktop\military_object_dataset\military_dataset.yaml")
assert best_weights.exists(), best_weights
assert data_yaml.exists(), data_yaml

# stabilize Windows/Jupyter
torch.cuda.empty_cache()
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
cv2.setNumThreads(0)
torch.set_num_threads(1)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# clear stale caches (esp. on OneDrive)
for cache in glob.glob(str(data_yaml.parent / "**" / "*.cache"), recursive=True):
    try:
        os.remove(cache)
    except Exception:
        pass

model = YOLO(str(best_weights))

metrics = model.val(
    data=str(data_yaml),
    imgsz=640,
    device=0,      # try 'cpu' once if it still hangs
    batch=4,       # drop to 2 if VRAM tight
    workers=0,     # critical on Windows/Jupyter
    half=True,
    plots=False,
    verbose=True,
)
print(metrics)


Ultralytics 8.3.235  Python-3.13.3 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 92 layers, 25,846,708 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 4.54.6 ms, read: 34.517.7 MB/s, size: 78.8 KB)
val: Scanning C:\Users\adars\OneDrive\Desktop\military_object_dataset\val\labels... 2941 images, 273 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2941/2941 151.3it/s 19.4s<0.0s
val: New cache created: C:\Users\adars\OneDrive\Desktop\military_object_dataset\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 736/736 1.7it/s 7:23<1.1s
                   all       2941       5081       0.69      0.529       0.57      0.379
    camouflage_soldier        385        510       0.76      0.753      0.777      0.429
                weapon        222        358      0.743      0.589       0.63      0.457
         military_tank        938       1787      0.814   

## 6) Inference on Test Set + Save YOLO TXT with Confidence

In [ ]:
import os, shutil
from pathlib import Path
from ultralytics import YOLO

# --- your actual dataset and weights ---
dataset_root = Path(r"C:\Users\adars\OneDrive\Desktop\military_object_dataset")  # dataset folder
test_images_dir = dataset_root / "test" / "images"
assert test_images_dir.exists(), f"❌ Test images directory not found: {test_images_dir}"

best_weights = Path(r"C:\Users\adars\Downloads\runs\detect\serve_smart_yolov85\weights\best.pt")
assert best_weights.exists(), f"❌ Weights not found: {best_weights}"

# --- clean up old prediction results ---
predict_dir = Path("runs/detect/predict")
if predict_dir.exists():
    shutil.rmtree(predict_dir)

# --- model setup ---
model = YOLO(str(best_weights))
IMG_SIZE = 640

# --- run prediction ---
pred_results = model.predict(
    source=str(test_images_dir),
    save=True,
    save_txt=True,
    save_conf=True,
    imgsz=IMG_SIZE,
    conf=0.25,
    iou=0.5,
    max_det=300,
    device=0,
    half=True,
    workers=0,     # prevent Windows hang
    verbose=True,
)

# --- print the actual folder Ultralytics saved to ---
actual_save_dir = Path(pred_results[0].save_dir)
print("✅ Inference complete.")
print("📁 Images saved to:", actual_save_dir)
print("📝 Labels saved under:", actual_save_dir / "labels")


NameError: name 'resolved_dataset_dir' is not defined

## 7) Package Submission Files
Ensures every test image has a matching `.txt` with correct basename, then zips the `labels/` folder.

In [ ]:

labels_dir = Path("runs/detect/predict/labels")
missing = []
for img_path in sorted(test_images_dir.glob("*")):
    if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp"]:
        continue
    txt = labels_dir / (img_path.stem + ".txt")
    if not txt.exists():
        missing.append(img_path.name)

if missing:
    print("⚠️ Missing TXT files for the following test images:")
    for m in missing:
        print(" -", m)
else:
    print("All test images have matching .txt files.")

zip_name = "results_yolo_txt.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in labels_dir.glob("*.txt"):
        zf.write(p, arcname=p.name)

print("Created:", zip_name)


## 8) (Optional) Export for CPU Devices

In [ ]:

export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)
# ONNX export for portability
cpu_export = val_model.export(format="onnx", opset=12, imgsz=IMG_SIZE)
print("Exported:", cpu_export)


## 9) Report Helper: Pull Key Numbers

In [ ]:

# Summarize key metrics for the 4-page report
from pprint import pprint
summary = {
    "project": PROJECT_NAME,
    "epochs": EPOCHS,
    "imgsz": IMG_SIZE,
    "batch": BATCH,
    "base_model": BASE_MODEL,
    "best_weights": str(best_weights),
}
print("Summary:")
pprint(summary)


---
**Next steps:**
1. Run through the notebook end‑to‑end.
2. Upload/point to the dataset.
3. Submit the generated `results_yolo_txt.zip` and fill the report template.

Good luck! 🚀